In [ ]:
# Projeto do TechChallenger da Fase 3
# Assistente médico para pacientes e médicos
# Consulta em modelo fine-tunado baseado em Llama com acrescimo de dados sintéticos produzidos pelo próprio projeto
# Execução de pipeline de tratamento dos dados sinstéticos usando Langchain
# Consulta ao LLM fine-tunado usando LangChain
# Consulta de dados armazenados nos documentos utilizando RAG
# Detecção de execução de funcionalidades orquestradas por LangGraph
# Loging de toda execução

In [ ]:
# ========== PRÉ-INSTALAÇÃO (executar uma vez no Google Colab) ==========
!pip install -q requests==2.32.4
!pip install -q transformers datasets peft accelerate bitsandbytes
!pip install -q langchain langgraph langchain-community langchain-huggingface
!pip install -q sentence-transformers faiss-cpu
!pip install -q pandas torch
print("Instalação concluída.")

In [ ]:
import requests
from langgraph.graph import StateGraph # O import correto do LangGraph

print(f"✅ Versão do requests: {requests.__version__}")

try:
    # Testando os dois pilares que você instalou
    import langchain_core
    import langgraph
    print("✅ LangChain e LangGraph carregados com sucesso!")
except ImportError as e:
    print(f"❌ Erro ao carregar: {e}")

In [ ]:
import requests
import langchain_community
import torch

print(f"✅ Requests version: {requests.__version__}")
print(f"✅ LangChain Community carregada!")
print(f"✅ GPU disponível: {torch.cuda.is_available()}")

In [ ]:
!pip install -U langchain

In [ ]:
# LangChain e Vetores
!pip install -U langchain langchain-community langchain-huggingface langgraph faiss-cpu

# Hugging Face e Fine-tuning
!pip install -U transformers datasets peft bitsandbytes accelerate

In [ ]:
!pip install -U langchain-text-splitters

In [ ]:
# Em vez de: from langchain.text_splitter import ...
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Agora deve funcionar!")

In [ ]:
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    print("Sucesso com o novo import!")
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    print("Sucesso com o import antigo!")

In [ ]:
!pip install -U langchain-text-splitters langchain-community

In [ ]:
# Novo padrão de importação
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# Garante todas as dependências do seu script de uma vez
!pip install -U langchain-text-splitters langchain-community langgraph faiss-cpu langchain-huggingface

In [ ]:
!pip install -U langchain-text-splitters langchain-community

In [ ]:
!pip install -U langchain-text-splitters

In [ ]:
# Tente este novo caminho, que é o padrão atual:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Se o erro persistir, use o caminho legado atualizado:
# from langchain.text_splitter import RecursiveCharacterTextSplitter

print("Agora sim: Sucesso!")

In [ ]:
import langchain
print(langchain.__version__)
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("Sucesso!")

In [ ]:
# ========== IMPORTS E CONFIGURAÇÃO ==========z
import os
import json
import re
import csv
from datetime import datetime
from pathlib import Path

# LangChain / RAG / Fine-tuning
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END

# Hugging Face (fine-tuning)
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Diretório base (ajustar no Colab se necessário)
BASE_DIR = Path("/content") if os.path.exists("/content") else Path(".")
DADOS_DIR = BASE_DIR / "dados_sinteticos_processados"
MODELO_DIR = BASE_DIR / "modelo_finetuned"
VECTORSTORE_PATH = BASE_DIR / "faiss_index"
LOG_OPERACOES = BASE_DIR / "logging-operações.csv"
LOG_CONSULTAS = BASE_DIR / "logging-consultas.csv"

for d in [DADOS_DIR, MODELO_DIR, VECTORSTORE_PATH]:
    d.mkdir(parents=True, exist_ok=True)
print("Imports e configuração OK.")

## Etapa 1 – Produção e carregamento de dados sintéticos

Os arquivos com prefixo `dados-sinteticos-` já foram criados no projeto. Esta célula carrega e expõe os dados para as etapas seguintes.

In [ ]:
#Chave de carga de dados sintéticos
LIGAR_DADOS_SINTETICOS = True

def listar_arquivos_dados_sinteticos(base_path=None):
    """Lista todos os arquivos com prefixo 'dados-sinteticos-' no diretório do projeto."""
    base = Path(base_path) if base_path else Path(".")
    if not base.is_absolute() and not base.exists():
        base = Path(os.getcwd())
    arquivos = list(base.glob("dados-sinteticos-*"))
    return [str(f) for f in arquivos]

def carregar_json(path):
    """Carrega um arquivo JSON."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def carregar_todos_dados_sinteticos(base_path=None):
    """Carrega todos os dados sintéticos (JSON) em um dicionário por contexto."""

    if LIGAR_DADOS_SINTETICOS:
        lista = listar_arquivos_dados_sinteticos(base_path)
    else:
        lista = None

    dados = {}
    for path in lista:
        nome = Path(path).stem
        if path.endswith(".json"):
            try:
                dados[nome] = carregar_json(path)
            except Exception as e:
                dados[nome] = {"erro": str(e)}
    return dados

# No Colab: fazer upload dos arquivos dados-sinteticos-*.json em /content ou montar Drive
# Local: os arquivos devem estar na mesma pasta do notebook
import os
if os.path.exists("/content"):
    os.chdir("/content")  # Colab
else:
    pass  # manter diretório atual em ambiente local

arquivos = None

if LIGAR_DADOS_SINTETICOS:
    arquivos = listar_arquivos_dados_sinteticos()
    print("Arquivos de dados sintéticos encontrados:", arquivos)

if arquivos:
    dados_brutos = carregar_todos_dados_sinteticos()
    for k, v in dados_brutos.items():
        print(f"  {k}: {len(v) if isinstance(v, list) else 'dict'} itens")
else:
    dados_brutos = {}
    print("Nenhum arquivo 'dados-sinteticos-*' encontrado. Faça upload no Colab ou coloque os JSONs na pasta do projeto.")

## Etapa 2 – Pipeline de anonimização

Substitui nomes de pessoas e dados de identificação (CPF, datas de nascimento, CRM) por identificadores anônimos.

In [ ]:
# Mapeamento nome/identificador -> anônimo (preenchido pelo pipeline)
MAPEAMENTO_ANONIMIZACAO = {}

# Chaveamento habilitação de anonimização
LIGAR_ANONIMIZACAO = True

def anonimizar_texto(texto):
    if LIGAR_ANONIMIZACAO:
        """Substitui CPF, CRM e nomes próprios por placeholders no texto."""
        if not isinstance(texto, str):
            return texto
        # CPF: xxx.xxx.xxx-xx -> [CPF_ANON]
        texto = re.sub(r"\d{3}\.\d{3}\.\d{3}-\d{2}", "[CPF_ANON]", texto)
        # CRM-XX 123456 -> [CRM_ANON]
        texto = re.sub(r"CRM-[A-Z]{2}\s*\d+", "[CRM_ANON]", texto)
        for nome_real, nome_anon in MAPEAMENTO_ANONIMIZACAO.items():
            texto = re.sub(re.escape(nome_real), nome_anon, texto, flags=re.IGNORECASE)
        return texto

def extrair_nomes_para_anonimizar(dados_dict):
    if LIGAR_ANONIMIZACAO:
        """Extrai nomes (paciente, medico) dos dados para criar mapeamento."""
        nomes = set()
        for nome_arq, conteudo in dados_dict.items():
            if isinstance(conteudo, list):
                for item in conteudo:
                    if isinstance(item, dict):
                        for k, v in item.items():
                            if v and isinstance(v, str) and k in ("paciente", "medico", "nome"):
                                nomes.add(v.strip())
            elif isinstance(conteudo, dict):
                for k, v in conteudo.items():
                    if k in ("paciente", "medico", "nome") and isinstance(v, str):
                        nomes.add(v.strip())
        return list(nomes)

def construir_mapeamento_anonimizacao(dados_dict):
    if LIGAR_ANONIMIZACAO:
        """Constrói mapa nome_real -> [TIPO_N] (ex: [PACIENTE_1], [MEDICO_1])."""
        global MAPEAMENTO_ANONIMIZACAO
        MAPEAMENTO_ANONIMIZACAO = {}
        nomes = extrair_nomes_para_anonimizar(dados_dict)
        for i, nome in enumerate(nomes, 1):
            MAPEAMENTO_ANONIMIZACAO[nome] = f"[PESSOA_{i}]"
        return MAPEAMENTO_ANONIMIZACAO

def anonimizar_objeto(obj):
    if LIGAR_ANONIMIZACAO:
        """Aplica anonimização recursiva em dict/list/str."""
        if isinstance(obj, str):
            return anonimizar_texto(obj)
        if isinstance(obj, list):
            return [anonimizar_objeto(x) for x in obj]
        if isinstance(obj, dict):
            return {k: anonimizar_objeto(v) for k, v in obj.items()}
        return obj

def executar_pipeline_anonimizacao(dados_brutos, salvar_em=None):
    if LIGAR_ANONIMIZACAO:
        """Pipeline completo: mapeamento + anonimização + opcionalmente salvar."""
        construir_mapeamento_anonimizacao(dados_brutos)
        dados_anon = anonimizar_objeto(dados_brutos)
        if salvar_em:
            Path(salvar_em).mkdir(parents=True, exist_ok=True)
            for nome, conteudo in dados_anon.items():
                path = Path(salvar_em) / f"{nome}.json"
                with open(path, "w", encoding="utf-8") as f:
                    json.dump(conteudo, f, ensure_ascii=False, indent=2)
        return dados_anon

# Executar após carregar dados
if 'dados_brutos' in dir() and dados_brutos:
    if LIGAR_ANONIMIZACAO:
        dados_anonimizados = executar_pipeline_anonimizacao(dados_brutos, DADOS_DIR)
        print("Anonimização concluída. Amostra:", list(dados_anonimizados.keys()))

## Etapa 3 – Integração e fine-tuning do modelo Llama

Preparação dos dados no formato de instrução, fine-tuning com LangChain/Hugging Face (PEFT/LoRA) e salvamento do modelo customizado.

In [ ]:
# Chaveamento habilitação de anonimização
LIGAR_ANONIMIZACAO = True

def dados_sinteticos_para_instrucoes(dados_anon):

    if LIGAR_ANONIMIZACAO:
        """Converte dados anonimizados em pares instrução/resposta para fine-tuning."""
        exemplos = []
        for nome_arq, conteudo in dados_anon.items():
            if not isinstance(conteudo, list):
                continue
            for item in conteudo:
                if isinstance(item, dict):
                    if "pergunta" in item and "resposta" in item:
                        exemplos.append({"instruction": item["pergunta"], "response": item["resposta"]})
                    elif "pergunta_medico" in item and "resposta" in item:
                        exemplos.append({"instruction": item["pergunta_medico"], "response": item["resposta"]})
                    elif "protocolo" in item and "descricao" in item:
                        inst = f"Qual o protocolo de {item.get('protocolo', '')}?"
                        exemplos.append({"instruction": inst, "response": item["descricao"]})
                    elif "exame" in item and "interpretacao" in item:
                        inst = f"Interpretar exame: {item.get('exame','')} - Resultado: {item.get('resultado','')}"
                        exemplos.append({"instruction": inst, "response": item["interpretacao"]})
                    elif "tipo" in item and "modelo" in item:
                        exemplos.append({"instruction": f"Modelo de {item['tipo']}", "response": item["modelo"]})
        return exemplos

def formatar_para_llama(instruction, response, template="default"):
    """Formata par instrução/resposta no estilo chat Llama."""
    if template == "default":
        text = f"<s>[INST] {instruction} [/INST] {response} </s>"
    else:
        text = f"Human: {instruction}\nAssistant: {response}"
    return text

def criar_dataset_finetuning(dados_anon, max_amostras=500):
    if LIGAR_ANONIMIZACAO:
        """Cria Dataset Hugging Face para treino."""
        exemplos = dados_sinteticos_para_instrucoes(dados_anon)[:max_amostras]
        if not exemplos:
            # fallback: criar exemplos genéricos a partir de FAQ/protocolos
            for nome_arq, conteudo in dados_anon.items():
                if isinstance(conteudo, list):
                    for item in conteudo:
                        if isinstance(item, dict):
                            for k, v in item.items():
                                if k in ("pergunta", "descricao", "resposta") and v:
                                    exemplos.append({"instruction": "Pergunta sobre saúde", "response": str(v)[:500]})
                    if len(exemplos) >= max_amostras:
                        break
        texts = [formatar_para_llama(e["instruction"], e["response"]) for e in exemplos]
        return Dataset.from_dict({"text": texts})

In [ ]:
# Modelo base open source Llama (TinyLlama para caber no Colab; pode trocar por meta-llama/Llama-2-7b-hf com mais RAM)
MODELO_BASE = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Chaveamento habilitação de anonimização
LIGAR_ANONIMIZACAO = True

def carregar_modelo_e_tokenizer(model_name=MODELO_BASE, use_4bit=True):
    """Carrega modelo e tokenizer para fine-tuning com quantização opcional."""
    bnb_config = BitsAndBytesConfig(load_in_4bit=use_4bit, bnb_4bit_compute_dtype="float16", bnb_4bit_quant_type="nf4") if use_4bit else None
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
    if use_4bit:
        model = prepare_model_for_kbit_training(model)
    return model, tokenizer

def executar_finetuning(dados_anon, modelo_base=MODELO_BASE, output_dir=None, num_epochs=2, batch_size=2):
    if LIGAR_ANONIMIZACAO:
        """Executa fine-tuning (LoRA) e salva o modelo customizado."""
        output_dir = output_dir or str(MODELO_DIR)
        ds = criar_dataset_finetuning(dados_anon)
        model, tokenizer = carregar_modelo_e_tokenizer(modelo_base)

        lora_config = LoraConfig(r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none")
        model = get_peft_model(model, lora_config)

    def tokenize(examples):
        return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

    ds_tokenized = ds.map(tokenize, batched=True, remove_columns=ds.column_names)

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,
        save_strategy="epoch",
        logging_steps=10,
        fp16=True,
    )
    trainer = Trainer(model=model, args=training_args, train_dataset=ds_tokenized)
    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return output_dir

# Descomente para rodar (demora e exige GPU no Colab):
# if 'dados_anonimizados' in dir():
#     caminho_modelo = executar_finetuning(dados_anonimizados, output_dir=str(MODELO_DIR))
#     print("Modelo salvo em:", caminho_modelo)

## Etapa 4 e 5 – Assistente com RAG e LangGraph

Construção do índice vetorial (RAG), grafo LangGraph para roteamento de intenções (sugerir tratamentos, análise de exames, alertas de risco, dúvidas gerais) e assistente com logs e regras de segurança.

In [ ]:
# Chaveamento habilitação de RAG
LIGAR_RAG = True

def documentos_para_rag(dados_anon):
    if LIGAR_RAG:
        """Converte dados anonimizados em lista de Document para RAG."""
        docs = []
        for nome_arq, conteudo in dados_anon.items():
            if isinstance(conteudo, list):
                for i, item in enumerate(conteudo):
                    if isinstance(item, dict):
                        texto = json.dumps(item, ensure_ascii=False)
                        docs.append(Document(page_content=texto, metadata={"fonte": nome_arq, "indice": i}))
            elif isinstance(conteudo, dict):
                docs.append(Document(page_content=json.dumps(conteudo, ensure_ascii=False), metadata={"fonte": nome_arq}))
        return docs

def construir_vectorstore(dados_anon, persist_path=None):
    if LIGAR_RAG:
        """Cria FAISS a partir dos documentos e opcionalmente persiste."""
        docs = documentos_para_rag(dados_anon)
        splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
        splits = splitter.split_documents(docs)
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
        vectorstore = FAISS.from_documents(splits, embeddings)
        if persist_path:
            vectorstore.save_local(str(persist_path))
        return vectorstore

def log_operacao(operacao, detalhe="", arquivo=LOG_OPERACOES):
    """Registra operação em logging-operações.csv com data e hora (minuto e segundo)."""
    with open(arquivo, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if f.tell() == 0:
            w.writerow(["data", "hora", "minuto", "segundo", "operacao", "detalhe"])
        now = datetime.now()
        w.writerow([now.strftime("%Y-%m-%d"), now.strftime("%H"), now.strftime("%M"), now.strftime("%S"), operacao, detalhe])

def log_consulta(tipo_usuario, nome_usuario, pergunta, resposta_assistente, arquivo=LOG_CONSULTAS):
    """Registra consulta em logging-consultas.csv."""
    with open(arquivo, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if f.tell() == 0:
            w.writerow(["data", "hora", "minuto", "segundo", "tipo_usuario", "nome_usuario", "pergunta", "resposta_assistente"])
        now = datetime.now()
        w.writerow([now.strftime("%Y-%m-%d"), now.strftime("%H:%M:%S"), now.strftime("%M"), now.strftime("%S"), tipo_usuario, nome_usuario, pergunta[:500], resposta_assistente[:1000]])

In [ ]:
import re
from typing import TypedDict, Literal

class EstadoAssistente(TypedDict):
    pergunta: str
    tipo_usuario: Literal["paciente", "medico"]
    nome_usuario: str
    intencao: str  # "tratamentos", "analise_exames", "alertas_risco", "duvida_geral"
    contexto_rag: str
    resposta: str
    fonte: str

def classificar_intencao(pergunta: str) -> str:
    """Classifica a intenção do usuário a partir do texto (simplificado; pode usar LLM depois)."""
    p = pergunta.lower()
    if any(x in p for x in ["tratamento", "procedimento", "como tratar", "terapia", "protocolo"]):
        return "tratamentos"
    if any(x in p for x in ["exame", "resultado", "laudo", "hemograma", "glicemia", "creatinina", "interpretar"]):
        return "analise_exames"
    if any(x in p for x in ["risco", "grave", "alerta", "urgente", "perigo", "crítico"]):
        return "alertas_risco"
    return "duvida_geral"

def node_classificar(state: EstadoAssistente) -> EstadoAssistente:
    state["intencao"] = classificar_intencao(state["pergunta"])
    log_operacao("classificacao_intencao", state["intencao"])
    return state

def node_buscar_rag(state: EstadoAssistente, vectorstore) -> EstadoAssistente:
    if vectorstore is None:
        state["contexto_rag"] = ""
        state["fonte"] = "modelo"
        return state

    # Extrair palavras-chave da pergunta
    query_clean = re.sub(r'[^\w\s]', '', state["pergunta"]).lower()
    query_keywords = [kw for kw in query_clean.split() if len(kw) >= 3] # Alterado de > 3 para >= 3

    docs = vectorstore.similarity_search(state["pergunta"], k=5) # Aumentar k para ter mais docs para filtrar

    filtered_docs = []
    # Filtrar documentos que contêm pelo menos uma palavra-chave relevante da consulta
    for doc in docs:
        doc_content_lower = doc.page_content.lower()
        # Considerar uma correspondência se a pergunta inteira (ou uma parte significativa) estiver presente
        # ou se várias palavras-chave estiverem presentes.
        # A pergunta completa em minúsculas deve estar no documento OU pelo menos uma palavra-chave deve estar.
        if state["pergunta"].lower() in doc_content_lower or \
           any(keyword in doc_content_lower for keyword in query_keywords):
            filtered_docs.append(doc)

    # Se após a filtragem não houver documentos, o filtered_docs permanecerá vazio.
    # Isso garante que não serão usados documentos irrelevantes como fallback.
    if filtered_docs:
        filtered_docs = filtered_docs[:3] # Limitar a 3 documentos mais relevantes após a filtragem

    state["contexto_rag"] = "\n\n".join(d.page_content for d in filtered_docs)
    state["fonte"] = "RAG: " + ", ".join(d.metadata.get("fonte", "doc") for d in filtered_docs) if filtered_docs else "Nenhuma fonte específica encontrada"
    return state

def node_gerar_resposta(state: EstadoAssistente, llm_inference_fn) -> EstadoAssistente:
    aviso = " Lembre-se: não prescrevo medicamentos nem substituo o médico. Sempre confirme com seu médico de acompanhamento ou médico de plantão."
    prompt = f"Pergunta ({state['tipo_usuario']}, {state['nome_usuario']}): {state['pergunta']}\nContexto (use para enriquecer): {state['contexto_rag'][:2000]}\nIntenção detectada: {state['intencao']}. Responda de forma útil e segura.{aviso}"
    if llm_inference_fn:
        resp = llm_inference_fn(prompt)
    else:
        # Resposta baseada no contexto RAG quando disponível
        ctx = (state["contexto_rag"] or "").strip()
        if not ctx:
            resp = f"Com base na intenção '{state['intencao']}': Não foram encontrados trechos específicos nos documentos para esta pergunta. "
        else:
            resp = f"Com base na intenção '{state['intencao']}': {ctx[:800]}" # Limita o contexto para não exceder o token
        resp += " Confirme sempre com seu médico para condutas e medicamentos." + aviso
    state["resposta"] = resp + f"\n\n[Fonte da informação: {state['fonte']}]."
    return state

In [ ]:
def construir_grafo_assistente(vectorstore, llm_inference_fn):
    """Monta o grafo LangGraph: classificar -> buscar RAG -> gerar resposta."""
    def buscar(state):
        return node_buscar_rag(state, vectorstore)

    def gerar(state):
        return node_gerar_resposta(state, llm_inference_fn)

    workflow = StateGraph(EstadoAssistente)
    workflow.add_node("classificar", node_classificar)
    workflow.add_node("buscar_rag", buscar)
    workflow.add_node("gerar_resposta", gerar)
    workflow.set_entry_point("classificar")
    workflow.add_edge("classificar", "buscar_rag")
    workflow.add_edge("buscar_rag", "gerar_resposta")
    workflow.add_edge("gerar_resposta", END)
    return workflow.compile()

def aplicar_regras_seguranca(texto: str) -> str:
    """Garante que a resposta inclua orientação de não prescrever e confirmar com médico."""
    if "confirmar" not in texto.lower() and "médico" not in texto.lower():
        texto += " Recomendo confirmar as informações com seu médico de acompanhamento ou médico de plantão para os próximos passos."
    if "medicamento" in texto.lower() or "remédio" in texto.lower():
        texto += " Não sugiro medicamentos; oriente-se sempre com o médico que acompanha o caso ou o médico de plantão."
    return texto

def inferencia_llm_simulada(prompt: str, modelo_path=None) -> str:
    """Inferência com modelo fine-tunado (ou simulada se modelo não disponível)."""
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM
        path = modelo_path or str(MODELO_DIR)
        if Path(path).exists():
            tokenizer = AutoTokenizer.from_pretrained(path)
            model = AutoModelForCausalLM.from_pretrained(path, device_map="auto")
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            out = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7)
            return tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()
    except Exception as e:
        log_operacao("erro_inferencia", str(e))
    return f"Resposta baseada no contexto e na intenção. [Modelo não carregado ou erro: use RAG]. Sempre consulte seu médico para condutas."

In [ ]:
def executar_assistente(pergunta: str, tipo_usuario: str, nome_usuario: str, grafo) -> str:
    """
    Executa o fluxo do assistente: grafo (classificar -> RAG -> resposta), aplica regras de segurança e loga.
    Nunca prescreve; sempre orienta a confirmar com médico.
    """
    estado_inicial = {
        "pergunta": pergunta,
        "tipo_usuario": tipo_usuario,
        "nome_usuario": nome_usuario,
        "intencao": "",
        "contexto_rag": "",
        "resposta": "",
        "fonte": "",
    }
    resultado = grafo.invoke(estado_inicial)
    resposta = resultado.get("resposta", "")
    resposta = aplicar_regras_seguranca(resposta)
    log_consulta(tipo_usuario, nome_usuario, pergunta, resposta)
    log_operacao("consulta_assistente", f"{tipo_usuario}/{nome_usuario}")
    return resposta

### Interface do assistente (campos de entrada)

Execute a célula abaixo para rodar o assistente: pergunta se é paciente ou médico, solicita o nome e processa a pergunta via LangGraph + RAG.

In [ ]:
def rodar_assistente_interativo(dados_anon=None, usar_modelo_finetuned=False):
    """
    Fluxo completo: pergunta se é paciente ou médico, pede o nome, depois aceita perguntas.
    Usa dados anonimizados para RAG e opcionalmente o modelo fine-tunado.
    """
    dados = dados_anon if dados_anon is not None else (dados_anonimizados if "dados_anonimizados" in dir() else {})
    if not dados:
        print("Nenhum dado anonimizado carregado. Carregue os dados sintéticos e execute a anonimização antes.")
        return

    vectorstore = construir_vectorstore(dados, persist_path=VECTORSTORE_PATH)
    llm_fn = (lambda p: inferencia_llm_simulada(p, str(MODELO_DIR))) if usar_modelo_finetuned else None
    grafo = construir_grafo_assistente(vectorstore, llm_fn)

    tipo_usuario = input("Você é paciente ou médico? (paciente/medico): ").strip().lower() or "paciente"
    if tipo_usuario not in ("paciente", "medico"):
        tipo_usuario = "paciente"
    nome_usuario = input("Qual o seu nome (ou do paciente)?: ").strip() or "Não informado"
    log_operacao("inicio_sessao", f"{tipo_usuario} - {nome_usuario}")

    while True:
        pergunta = input("\nSua pergunta (ou Enter para encerrar): ").strip()
        if not pergunta:
            break
        resposta = executar_assistente(pergunta, tipo_usuario, nome_usuario, grafo)
        print("\n--- Assistente Médico ---")
        print(resposta)
        print("------------------------")
    log_operacao("fim_sessao", nome_usuario)
    print("Sessão encerrada. Logs em logging-operações.csv e logging-consultas.csv.")

In [ ]:
# ========== EXECUÇÃO COMPLETA (descomente e execute em sequência no Colab) ==========
# 1) Garantir que os arquivos dados-sinteticos-*.json estão no diretório (upload no Colab ou use o caminho do projeto)
# 2) Carregar dados (use o caminho onde estão os JSONs)
# dados_brutos = carregar_todos_dados_sinteticos("/content")  # Colab
# dados_brutos = carregar_todos_dados_sinteticos()  # local
# 3) Anonimizar
# dados_anonimizados = executar_pipeline_anonimizacao(dados_brutos, DADOS_DIR)
# 4) (Opcional) Fine-tuning - exige GPU e tempo
# executar_finetuning(dados_anonimizados, output_dir=str(MODELO_DIR))
# 5) Rodar o assistente interativo (RAG + LangGraph)
# rodar_assistente_interativo(usar_modelo_finetuned=False)

# Teste rápido sem input (uma pergunta fixa):
if "dados_anonimizados" in dir() and dados_anonimizados:
    vs = construir_vectorstore(dados_anonimizados)
    grafo = construir_grafo_assistente(vs, None)
    #r = executar_assistente("O que significa glicemia elevada?", "paciente", "Teste", grafo)
    r = executar_assistente("Estou com febre de 40 graus o que devo fazer?", "paciente", "Bruno", grafo)
    print("Resposta de teste:", r[:400])

In [ ]:
if 'grafo' in locals() and 'vectorstore' in locals():
    print("Processando a pergunta 'HIV'...")
    # Usando os valores de tipo_usuario e nome_usuario do kernel, se disponíveis
    # Caso contrário, usaremos valores padrão para o teste
    current_tipo_usuario = locals().get('tipo_usuario', 'paciente')
    current_nome_usuario = locals().get('nome_usuario', 'Teste Automático')

    resposta_hiv = executar_assistente("HIV", current_tipo_usuario, current_nome_usuario, grafo)
    print("\n--- Assistente Médico ---")
    print(resposta_hiv)
    print("-------------------------")
else:
    print("O assistente não está completamente inicializado. Por favor, execute a célula principal (cc2a97ec) novamente.")

In [29]:
from IPython.display import clear_output

## PROGRAMA PRINCIPAL ## ASSISTENTE MÉDICO INTERATIVO ## 2026 - TECHCHALLENGER - FASE 3 - FIAP

clear_output(wait=True)
print("Inicializando o assistente médico interativo...")

# Certifique-se de que 'dados_anonimizados' está disponível do pipeline anterior
if "dados_anonimizados" not in dir() or not dados_anonimizados:
    print("Nenhum dado anonimizado carregado. Carregue os dados sintéticos e execute a anonimização antes de rodar o assistente interativo.")
else:
    # Construir o vectorstore para RAG
    # Reconstruímos aqui para garantir que o vectorstore esteja atualizado ou carregado
    vectorstore = construir_vectorstore(dados_anonimizados, persist_path=VECTORSTORE_PATH)

    # Definir se usará o modelo fine-tunado (True/False)
    # Por padrão, manteremos como False, a menos que o fine-tuning tenha sido explicitamente executado.
    usar_modelo_finetuned = False # Altere para True se você executou o fine-tuning
    llm_fn = (lambda p: inferencia_llm_simulada(p, str(MODELO_DIR))) if usar_modelo_finetuned else None

    # Construir o grafo do assistente com o vectorstore e a função de inferência do LLM
    grafo = construir_grafo_assistente(vectorstore, llm_fn)

    print("\n\n Assistente pronto. Digite 'sair' a qualquer momento para encerrar.")

    tipo_usuario = input("\n Você é paciente ou médico? (paciente/medico): ").strip().lower() or "paciente"
    if tipo_usuario not in ("paciente", "medico"):
        tipo_usuario = "paciente"
    nome_usuario = input("Qual o seu nome (ou do paciente)?: ").strip() or "Não informado"
    log_operacao("inicio_sessao", f"{tipo_usuario} - {nome_usuario}")

    while True:
        pergunta = input("\nSua pergunta: ").strip()
        if not pergunta or pergunta.lower() == 'sair':
            print("Encerrando o assistente.")
            break

        print("Processando sua pergunta...")
        resposta = executar_assistente(pergunta, tipo_usuario, nome_usuario, grafo)

        print("\n--- Assistente Médico ---")
        print(resposta)
        print("-------------------------")

    log_operacao("fim_sessao", nome_usuario)
    print("Sessão encerrada. Logs em logging-operações.csv e logging-consultas.csv.")

Inicializando o assistente médico interativo...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.




 Assistente pronto. Digite 'sair' a qualquer momento para encerrar.

 Você é paciente ou médico? (paciente/medico): MEDICO
Qual o seu nome (ou do paciente)?: BRUNO

Sua pergunta: HIV
Processando sua pergunta...

--- Assistente Médico ---
Com base na intenção 'duvida_geral': {"protocolo": "Protocolo de Diagnóstico e Manejo Inicial de HIV", "instituicao": "Ministério da Saúde / Protocolo Nacional", "descricao": "Realização de testes rápidos. Se reagente, iniciar TARV imediatamente (esquema preferencial: TDF/3TC + DTG) e solicitar exames de base (CD4, CV, Genotipagem).", "indicacoes": "Pacientes com diagnóstico recente de infecção pelo HIV."}

{"paciente": "[PESSOA_8]", "cpf": "[CPF_ANON]", "data_nascimento": "12/05/1988", "exame": "Sorologia para HIV (ELISA)", "data_exame": "2024-03-10", "resultado": "Reagente para HIV-1 e HIV-2", "interpretacao": "Amostra reagente. Necessário teste confirmatório (Western Blot ou PCR) conforme protocolo.", "gravidade": "alta", "doenca_sugerida": "Infec

KeyboardInterrupt: Interrupted by user